In [1]:
import os
import json
from typing_extensions import TypedDict, Optional, List, Dict, Any
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

from IPython.display import display, Image

In [2]:
class ProductDescriptionGeneratorState(TypedDict):
    # Input
    product_id: str # ID of the product

    # Database info
    product_name: Optional[str] # The name of the product
    product_image_url: Optional[str] # The image URL of the product
    product_features: Optional[List[str]] # Features of product
    product_category: Optional[str] # Product category
    product_specifications: Optional[Dict[str, Any]] # Technical specifications of the product

    # Output
    product_features_from_image: Optional[str] # The decription of the product from the image
    product_description: Optional[str] # The main description of the product
    product_short_description: Optional[str] # Short summary for product
    # SEO - search engine optimization
    product_seo_title: Optional[str] # SEO optimized title for product page
    product_seo_description: Optional[str] # SEO optimized meta description
    product_keywords: Optional[List[str]] # Keywords for SEO purposes

#### Read products

In [3]:
current_dir = os.getcwd()
mock_data_path = os.path.join(current_dir, "mock_data", "mock_data.json")

assert os.path.exists(mock_data_path)

print(f'Read products from {mock_data_path}')

def find_product_details(
    state: ProductDescriptionGeneratorState,
) -> ProductDescriptionGeneratorState:
    assert os.path.exists(mock_data_path)

    with open(mock_data_path, "r") as data_file:
        products: List[ProductDescriptionGeneratorState] = json.load(data_file)

    assert len(products) > 0

    for product in products:
        if product["product_id"] == state["product_id"]:
            return product
    return {}

product_details_p001 = find_product_details({'product_id': 'P001'})
product_details_p001

Read products from /Users/dimoynwa/Development/AI-Bootcamp-With-LangGraph-and-Langchain/LangGraph Basics/Workflows/prompt_chaining/mock_data/mock_data.json


{'product_id': 'P001',
 'product_name': 'Apple iPhone 14 Pro',
 'product_image_url': 'https://www.359gsm.com/wp-content/uploads/2022/09/Apple-iPhone-14-Pro-iPhone-14-Pro-Max-space-black-220907_inline.jpg.large_2x.jpg',
 'product_features': ['6.1-inch Super Retina XDR OLED display',
  'A16 Bionic chip',
  '48 MP Pro camera system',
  'Dynamic Island & Always-On display',
  'Emergency SOS via satellite'],
 'product_category': 'Electronics',
 'product_specifications': {'screen_size': '6.1 inches',
  'resolution': '2556x1179 pixels',
  'storage_options': ['128 GB', '256 GB', '512 GB', '1 TB'],
  'weight': '206 g',
  'battery_life': 'Up to 23 hours video playback'},
 'product_price': 999.0,
 'product_availability': True}

#### Load environment variables

In [4]:
env_file = '../.env'
assert os.path.exists(env_file)

load_dotenv(env_file)

OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
assert OPENAI_API_KEY
print(f'----> OPENAI_API_KEY: {OPENAI_API_KEY[:3]}***{OPENAI_API_KEY[-3:]}')


LANGCHAIN_API_KEY = os.environ['LANGCHAIN_API_KEY']
assert LANGCHAIN_API_KEY
print(f'----> LANGCHAIN_API_KEY: {LANGCHAIN_API_KEY[:3]}***{LANGCHAIN_API_KEY[-3:]}')

----> OPENAI_API_KEY: sk-***tIA
----> LANGCHAIN_API_KEY: lsv***43b


#### Initialize model 

In [5]:
llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0.2)
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x11926a270>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11926acf0>, root_client=<openai.OpenAI object at 0x119268050>, root_async_client=<openai.AsyncOpenAI object at 0x11926aa50>, model_name='gpt-4.1-mini', temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'))

#### Image description

In [7]:
system_instructions = """
You are a sales expert. Based on the provided image you should generate a precised description of the image for marketing purpose.

REQUIREMENTS:
- Maximum words: 100
- Focus on physical appearance, visible features, and distinguishing elements.
- Ensure the extracted information is factual and directly observable in the image.
- Return information in bullet-points.
"""

messages = [
    SystemMessage(content=system_instructions),
    HumanMessage(content=[
        {
        "type": "image",
        "source_type": "url",
        "url": product_details_p001['product_image_url']
        }
    ])
]

display(Image(url=product_details_p001['product_image_url']))
image_attributes_response = llm.invoke(input=messages)
image_attributes_response.pretty_print()

================================== Ai Message ==================================

- Sleek, modern smartphone with a matte black finish.
- Features a flat-edge design with polished sides.
- Rear side showcases a triple-camera system with large lenses and a flash.
- Front display has minimal bezels and a pill-shaped cutout for the front camera and sensors.
- Reflective Apple logo centered on the back.
- Smooth, seamless glass front and back surfaces.
- Volume and mute buttons located on the left side.


In [8]:
product_details_p001['product_features_from_image'] = image_attributes_response.content
product_details_p001

{'product_id': 'P001',
 'product_name': 'Apple iPhone 14 Pro',
 'product_image_url': 'https://www.359gsm.com/wp-content/uploads/2022/09/Apple-iPhone-14-Pro-iPhone-14-Pro-Max-space-black-220907_inline.jpg.large_2x.jpg',
 'product_features': ['6.1-inch Super Retina XDR OLED display',
  'A16 Bionic chip',
  '48 MP Pro camera system',
  'Dynamic Island & Always-On display',
  'Emergency SOS via satellite'],
 'product_category': 'Electronics',
 'product_specifications': {'screen_size': '6.1 inches',
  'resolution': '2556x1179 pixels',
  'storage_options': ['128 GB', '256 GB', '512 GB', '1 TB'],
  'weight': '206 g',
  'battery_life': 'Up to 23 hours video playback'},
 'product_price': 999.0,
 'product_availability': True,
 'product_features_from_image': '- Sleek, modern smartphone with a matte black finish.\n- Features a flat-edge design with polished sides.\n- Rear side showcases a triple-camera system with large lenses and a flash.\n- Front display has minimal bezels and a pill-shaped cuto

#### Detailed desciption 

In [9]:
detailed_message_prompt = """
You are a marketting expert. You will receive a product details and you need to create a full detailed description.
WHAT YOU RECEIVE:
- product name
- product features
- producs specifications
- visibile features

REQUIREMENTS:
- Maximum word lengths: 500
- Do NOT focus too much on visible features
- Tone should be formal
"""

product_details = f"""
Name: {product_details_p001["product_name"]},
Features: {product_details_p001["product_features"]},
Specifications: {product_details_p001["product_specifications"]}
Visibility features: {product_details_p001["product_features_from_image"]}
"""

input_messages = [
    SystemMessage(content=detailed_message_prompt),
    HumanMessage(content=product_details)
]

description_response = llm.invoke(input=input_messages)
description_response.pretty_print()

================================== Ai Message ==================================

The Apple iPhone 14 Pro represents the pinnacle of smartphone innovation, combining cutting-edge technology with sophisticated design to deliver an unparalleled user experience. At its core lies the powerful A16 Bionic chip, engineered to provide exceptional performance and efficiency. This advanced processor ensures seamless multitasking, rapid app launches, and smooth graphics rendering, making the device ideal for both everyday use and demanding applications.

The iPhone 14 Pro features a stunning 6.1-inch Super Retina XDR OLED display, offering vibrant colors, deep blacks, and remarkable brightness. With a resolution of 2556 by 1179 pixels, the screen delivers crisp and detailed visuals, enhancing everything from video playback to gaming and professional photo editing. The display incorporates innovative technologies such as Dynamic Island and Always-On display, which provide intuitive notifications a

In [10]:
product_details_p001['product_description'] = description_response.content
product_details_p001

{'product_id': 'P001',
 'product_name': 'Apple iPhone 14 Pro',
 'product_image_url': 'https://www.359gsm.com/wp-content/uploads/2022/09/Apple-iPhone-14-Pro-iPhone-14-Pro-Max-space-black-220907_inline.jpg.large_2x.jpg',
 'product_features': ['6.1-inch Super Retina XDR OLED display',
  'A16 Bionic chip',
  '48 MP Pro camera system',
  'Dynamic Island & Always-On display',
  'Emergency SOS via satellite'],
 'product_category': 'Electronics',
 'product_specifications': {'screen_size': '6.1 inches',
  'resolution': '2556x1179 pixels',
  'storage_options': ['128 GB', '256 GB', '512 GB', '1 TB'],
  'weight': '206 g',
  'battery_life': 'Up to 23 hours video playback'},
 'product_price': 999.0,
 'product_availability': True,
 'product_features_from_image': '- Sleek, modern smartphone with a matte black finish.\n- Features a flat-edge design with polished sides.\n- Rear side showcases a triple-camera system with large lenses and a flash.\n- Front display has minimal bezels and a pill-shaped cuto

#### Short message 

In [11]:
title_prompt = f"""
Based on product name and product detailed description generate a marketting title.

Name: {product_details_p001['product_name']}
Description: {product_details_p001['product_description']}

REQUIREMENTS:
- Up to 20 words
"""

title_response = llm.invoke(input=[title_prompt])
title_response.pretty_print()

================================== Ai Message ==================================

Apple iPhone 14 Pro – Ultimate Performance, Stunning Display, Pro-Grade Camera, Extended Battery, and Advanced Safety Features


In [12]:
product_details_p001['product_short_description'] = title_response.content
product_details_p001

{'product_id': 'P001',
 'product_name': 'Apple iPhone 14 Pro',
 'product_image_url': 'https://www.359gsm.com/wp-content/uploads/2022/09/Apple-iPhone-14-Pro-iPhone-14-Pro-Max-space-black-220907_inline.jpg.large_2x.jpg',
 'product_features': ['6.1-inch Super Retina XDR OLED display',
  'A16 Bionic chip',
  '48 MP Pro camera system',
  'Dynamic Island & Always-On display',
  'Emergency SOS via satellite'],
 'product_category': 'Electronics',
 'product_specifications': {'screen_size': '6.1 inches',
  'resolution': '2556x1179 pixels',
  'storage_options': ['128 GB', '256 GB', '512 GB', '1 TB'],
  'weight': '206 g',
  'battery_life': 'Up to 23 hours video playback'},
 'product_price': 999.0,
 'product_availability': True,
 'product_features_from_image': '- Sleek, modern smartphone with a matte black finish.\n- Features a flat-edge design with polished sides.\n- Rear side showcases a triple-camera system with large lenses and a flash.\n- Front display has minimal bezels and a pill-shaped cuto